
#### Course: DATA3950 - Introduction to Machine Learning and Artificial Intelligence 
#### Group 2: Sam, Harmeet, Adnan


### Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):

1. Use NLP techniques to process the training data.
2. Train model(s) to predict which class(es) each comment is in.
   - A comment can belong to any number of classes, including none.
3. Generate predictions for each of the comments in the test data.
4. Write your test data predictions to a CSV file, which will be scored.

You can use any models and NLP libraries you'd like.

---

### Project Overview – NLP Angry Comment Classification

#### Objective

The goal of this project is to build a Natural Language Processing (NLP) model that classifies comments into different categories of offensive language.

Each comment can belong to one or more of the following categories:

- Toxic
- Severe Toxic
- Obscene
- Threat
- Insult
- Identity Hate

A comment may belong to multiple categories or none at all.

---

#### What Type of Problem Is This?

This is a **Multi-Label Classification Problem**.

That means:

- We are not predicting just one class.
- We are predicting multiple independent categories.
- Each label is treated separately (0 or 1).

---

#### What We Will Do

1. Clean and process text using NLP techniques
2. Convert text into numerical features (TF-IDF)
3. Train classification models
4. Predict on test data
5. Export predictions in the required CSV format

In [ ]:
# ============================================
#  IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

# NLP & Text Processing
from sklearn.feature_extraction.text import TfidfVectorizer

# Model
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report

# Train / Validation Split
from sklearn.model_selection import train_test_split  # 

# File handling
import os
import keras

In [9]:
import pandas as pd
import keras

# Train Data
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
file_1 = keras.utils.get_file("train.csv.zip", url_1)
train_df = pd.read_csv(file_1)

# Test Data
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"
file_2 = keras.utils.get_file("test.csv.zip", url_2)
test_df = pd.read_csv(file_2)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (159571, 8)
Test shape: (153164, 2)


Files are downloaded automatically and stored locally. This ensures all group members can run the notebook without path errors.

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

In [ ]:
#train_df = pd.read_csv("train.csv.zip")
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


comment_text contains the actual text

Columns toxic → identity_hate are labels (0 or 1)

Each label is independent

Check for missing values; none are present

## Test Data

In [ ]:
#test_df = pd.read_csv("test.csv")
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \n\n The title is fine as it is...
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.



- This is the data we want to predict: the category of each comment.
- IDs in test file may not match training IDs; for submission, we assign sequential IDs 1,2,3… 

## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [ ]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [ ]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

#### Define Features and Labels

In [ ]:
# Features (input) and Labels (output)
X = train_df["comment_text"]
y = train_df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]]

# Test comments for prediction
X_test = test_df["comment_text"]



- Preparing the text for the model.
- Each comment may have multiple labels (0 or 1)

#### Train / Validation Split

In [19]:
# Split training data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Validation size:", X_valid.shape)

Training size: (127656,)
Validation size: (31915,)


**Observation:**
- The training set has 127,656 comments used to teach the model, and the validation set has 31,915 comments used to check performance. This 80/20 split helps ensure the model learns well without overfitting.

#### TF-IDF Vectorization

In [20]:
# Convert text into numerical features using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)

TF-IDF training shape: (127656, 20000)


**Observation:**
- The TF-IDF matrix has shape (127,656, 20,000), meaning we have 127,656 comments and each comment is represented by 20,000 features (the top 20,000 most important words).  
- TF-IDF (Term Frequency – Inverse Document Frequency): converts text into numbers so the model can understand and learn from it, giving more weight to important or unique words and less to common words like "the" or "and".

#### Train Multi-Label Classifier

In [21]:
# One classifier per label
model = OneVsRestClassifier(LogisticRegression(max_iter=2000))
model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")

Model trained successfully.


**Observation:**
- The multi-label classifier has been trained on the TF-IDF features.  
- Each label (toxic, severe_toxic, obscene, threat, insult, identity_hate) has its own classifier.  
- The model is now ready to predict labels on the validation set and test comments.

#### Validate Model Performance

In [22]:
# Predict on validation set
y_pred_valid = model.predict(X_valid_tfidf)

# Evaluate
print(classification_report(y_valid, y_pred_valid))

              precision    recall  f1-score   support

           0       0.91      0.60      0.72      3056
           1       0.60      0.25      0.35       321
           2       0.92      0.63      0.74      1715
           3       0.47      0.11      0.18        74
           4       0.82      0.49      0.62      1614
           5       0.71      0.14      0.24       294

   micro avg       0.88      0.54      0.67      7074
   macro avg       0.74      0.37      0.47      7074
weighted avg       0.86      0.54      0.66      7074
 samples avg       0.05      0.05      0.05      7074



c:\Users\SamAdeoye\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SamAdeoye\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SamAdeoye\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


**Observation:**
- The validation report shows precision, recall, and F1-score for each label.  
- Labels like `toxic` and `obscene` have relatively high F1-scores, meaning the model predicts them well.  
- Labels like `threat` and `identity_hate` have lower F1-scores, showing the model struggles with these rare categories.  
- Overall, micro F1-score is 0.67, indicating moderate performance.  
- **This lets us see which labels the model predicts well and which need improvement.**

#### Predict Test Data & Prepare Submission

In [23]:
# Predict test set
test_predictions = model.predict(X_test_tfidf)

# Build submission DataFrame
submission = pd.DataFrame(test_predictions, columns=y.columns)
submission.insert(0, "id", range(1, len(submission)+1))

# Save submission CSV
os.makedirs("output", exist_ok=True)
submission.to_csv("output/out.csv", index=False)
print("Submission saved as output/out.csv")

Submission saved as output/out.csv


**Observation:**
- The model has predicted labels for all test comments.  
- Predictions are saved in `output/out.csv` with the correct structure: first column = ID, followed by the six label columns.  
- This CSV is ready to be submitted to Moodle, matching the required format for evaluation.

### Observations & Improvements

**Observations:**
- The model predicts common labels like `toxic` and `obscene` well.  
- Rare categories like `threat` and `identity_hate` are harder to predict, likely due to fewer examples in the training data.  
- Micro F1-score is moderate, indicating overall performance is okay but can improve.

**Ideas to Improve Accuracy:**
- **Hyperparameter Tuning:** Adjust Logistic Regression parameters (C, max_iter) or try other classifiers like Random Forest or XGBoost.  
- **Text Preprocessing:** Remove emojis, URLs, or apply stemming/lemmatization to normalize words.  
- **Advanced NLP Models:** Use pre-trained embeddings (Word2Vec, GloVe) or transformer-based models like BERT for better context understanding.  
- **Data Augmentation:** Increase samples for rare classes using techniques like back-translation or oversampling.

In [26]:
# ============================================
#  Quick CSV Preview Check
# ============================================

import pandas as pd

# Load the CSV we just saved
submission = pd.read_csv("output/out.csv")

#  Show first 5 rows
print("First 5 rows of the submission CSV:")
display(submission.head())

#  Check column names
print("\nColumn names in CSV:")
print(submission.columns.tolist())

#  Check number of rows
print("\nNumber of rows in CSV:")
print(submission.shape)

#  Check predicted counts for each label
print("\nPredicted counts for each label:")
print(submission.iloc[:, 1:].sum())

First 5 rows of the submission CSV:


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,1,0,1,0,1,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0



Column names in CSV:
['id', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Number of rows in CSV:
(153164, 7)

Predicted counts for each label:
toxic            24807
severe_toxic      1087
obscene          13883
threat             239
insult           10118
identity_hate      861
dtype: int64


**Observation: Submission CSV Preview**

- The first 5 rows show that each test comment has a unique ID and predictions for all six labels.  
- Column order is correct: `id, toxic, severe_toxic, obscene, threat, insult, identity_hate`.  
- Total number of rows matches the number of test comments (153,164), so no comments are missing.  
- Predicted counts show how many comments were marked positive for each label:
  - Most common: `toxic` (24,807) and `obscene` (13,883)  
  - Least common: `threat` (239) and `identity_hate` (861)  
-  This confirms that the CSV is structured correctly and ready for Moodle submission.

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>